# 13 — CNN rung 4, experiment 5: NL-means denoising

**Decision this feeds** (`RESOURCES.md`, Boulkrinat et al. 2025's
preprocessing pipeline step 3): the last of the 5 rung-4 experiments,
and the riskiest to validate. `data.py::denoise_volume`
(`config.USE_NLM_DENOISING`, TDD'd 2026-09-09) applies patch-wise
Non-Local-Means denoising to the resampled+cropped volume before
intensity normalization — **this is the only rung-4 experiment that
changes `data.py`'s shared preprocessing pipeline itself**, not just a
training-loop knob. If it ever gets promoted, `submission_src/main.py`
runs this exact code at inference time too.

**Two real costs this experiment has that experiments 1-4 didn't:**
1. `config.USE_NLM_DENOISING=True` changes the volume-cache fingerprint
   (`cache.CachedVolumeStore`), so the existing warm cache from rungs
   2/3 **cannot** be reused — this notebook forces a full rebuild.
   `notebooks/06_cnn_rung2.ipynb` measured the non-denoised rebuild at
   ~2635s (44 min) for 1362 volumes; denoising adds real per-volume cost
   on top of that, unmeasured until cell 2 below.
2. If this experiment ever gets promoted to production, NL-means
   denoising also runs **once per test volume during the actual
   competition submission** (`submission_src/main.py` calls
   `data.load_volume`, which would then always denoise) — a real risk
   to the 3-hour inference budget worth weighing against any log-loss
   gain, not just the training-time cost.

**Dependency note**: `data.denoise_volume` deliberately does NOT use
`skimage.restoration.estimate_sigma` — checked the DrivenData runtime's
`uv.lock` and its hard dependency, PyWavelets, is absent even though
`scikit-image` itself is present. Uses a scipy-only Laplacian-MAD sigma
estimator instead (see `data.py`'s docstring).

**Cell order matters here**: cell 2 times denoising on a small real
sample *before* committing to the full cache rebuild in cell 3 — look at
its output and decide whether to continue before running the rest of
this notebook.

**Gate**: same as experiments 1-4 — nested-CV + paired bootstrap against
the **current validated CNN** (rung 3, `README.md` 2026-09-09:
mean=0.4520, sd=0.0109). No hyperparameter search (Boulkrinat et al.'s
`patch_size=3, patch_distance=5`, already the defaults in
`data.denoise_volume`).

**Data handling**: this notebook loads real `.nii.gz` volumes and
row-level labels throughout, so per the AI-assistant data rule
(`README.md`) it is **[RUN ME]** — run it yourself, share back only the
printed aggregate numbers, never any per-row output.

In [1]:
# [RUN ME] -- loads real row-level labels (no pixel data yet -- that's
# cells 2-3). Does NOT touch the volume cache.
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import nibabel as nib
import numpy as np
import pandas as pd
import torch

import cache
import config
import data
import dataset
import evaluate
import model
import train as train_mod

labels_df = pd.read_csv(config.TRAIN_LABELS_PATH)
family_df = pd.read_csv(config.DATA_PROCESSED / "baseline_features.csv")[
    [config.UID_COLUMN, "inplane_family"]
]
labeled_df = labels_df.merge(family_df, on=config.UID_COLUMN, how="inner").reset_index(drop=True)

uids = labeled_df[config.UID_COLUMN].tolist()
labels = labeled_df[config.TARGET_COLUMN].tolist()
families = labeled_df["inplane_family"].tolist()
print(f"{len(uids)} labeled volumes")

1362 labeled volumes


In [2]:
# [RUN ME] -- STOP AND READ THE OUTPUT before running the next cell.
# Times denoise_volume on a small real sample (resample+crop only, the
# pre-denoise steps -- no cache involved) to estimate the added cost of
# rebuilding the full 1362-volume cache WITH denoising, before
# committing to that rebuild.
SAMPLE_SIZE = 15
sample_uids = uids[:SAMPLE_SIZE]

resample_crop_times, denoise_times = [], []
for uid in sample_uids:
    img = nib.load(str(config.NIFTI_DIR / f"{uid}.nii.gz"))
    t0 = time.time()
    resampled, _ = data.resample_to_spacing(img.get_fdata(), img.affine, config.TARGET_SPACING)
    cropped = data.crop_or_pad(resampled, config.TARGET_SPACING, config.CROP_CENTER_MM, config.TARGET_SHAPE)
    resample_crop_times.append(time.time() - t0)

    t0 = time.time()
    data.denoise_volume(cropped)
    denoise_times.append(time.time() - t0)

mean_resample_crop = np.mean(resample_crop_times)
mean_denoise = np.mean(denoise_times)
print(f"resample+crop: {mean_resample_crop * 1000:.0f} ms/volume (n={SAMPLE_SIZE} sample)")
print(f"denoise_volume: {mean_denoise * 1000:.0f} ms/volume (n={SAMPLE_SIZE} sample)")
print(f"denoising adds {mean_denoise / mean_resample_crop:.1f}x the resample+crop cost per volume")
print(f"\nestimated FULL cache rebuild time with denoising, {len(uids)} volumes: "
      f"{(mean_resample_crop + mean_denoise) * len(uids) / 60:.0f} min "
      f"(non-denoised rebuild measured at ~44 min in notebook 06 for comparison)")

resample+crop: 271 ms/volume (n=15 sample)
denoise_volume: 1100 ms/volume (n=15 sample)
denoising adds 4.1x the resample+crop cost per volume

estimated FULL cache rebuild time with denoising, 1362 volumes: 31 min (non-denoised rebuild measured at ~44 min in notebook 06 for comparison)


In [3]:
# [RUN ME] -- only run this after reading cell 2's timing estimate.
# Enables denoising and builds the volume cache for all 1362 volumes.
# This is the expensive one-time cost cell 2 estimated.
config.USE_NLM_DENOISING = True

# A separate cache_dir (not the shared "volume_cache/" rungs 2/3/9-12
# use) -- belt-and-suspenders: this guarantees no risk of ever serving
# denoised volumes to a notebook expecting non-denoised ones (or vice
# versa), on top of the fingerprint mismatch that would already force a
# rebuild if this used the shared directory. USE_NLM_DENOISING is still
# included in the fingerprint below so re-running THIS notebook after
# ever changing it back to False also rebuilds correctly.
config_fingerprint = {
    "TARGET_SPACING": config.TARGET_SPACING,
    "CROP_SIZE_MM": config.CROP_SIZE_MM,
    "CROP_CENTER_MM": config.CROP_CENTER_MM,
    "TARGET_SHAPE": config.TARGET_SHAPE,
    "BACKGROUND_PERCENTILE": config.BACKGROUND_PERCENTILE,
    "BACKGROUND_MAX_FRACTION": config.BACKGROUND_MAX_FRACTION,
    "USE_NLM_DENOISING": config.USE_NLM_DENOISING,
}

cache_start = time.time()
volume_cache = cache.CachedVolumeStore(
    uids, cache_dir=config.DATA_PROCESSED / "volume_cache_denoised",
    config_fingerprint=config_fingerprint,
)
print(f"denoised cache {'reused' if volume_cache.was_reused else 'built'} in "
      f"{time.time() - cache_start:.1f}s for {len(uids)} volumes")

denoised cache built in 3984.5s for 1362 volumes


In [4]:
# [RUN ME] (no data access itself). Same helper as notebooks 06/07 --
# no special wiring needed here, since denoising already happened once
# when the cache was built (cell 3); every fold just reads from
# `volume_cache` like normal.
def train_and_score_nested(train_uids, train_labels, train_family,
                            outer_uids, batch_size, lr, seed,
                            epochs=config.EPOCHS, patience=config.PATIENCE,
                            inner_splits=10):
    inner_train_idx, inner_val_idx = evaluate.make_folds(
        train_labels, train_family, n_splits=inner_splits, random_state=seed
    )[0]

    def subset(idxs):
        return ([train_uids[i] for i in idxs], [train_labels[i] for i in idxs])

    inner_train_uids, inner_train_labels = subset(inner_train_idx)
    inner_val_uids, inner_val_labels = subset(inner_val_idx)

    inner_train_ds = dataset.DatParkinsonDataset(inner_train_uids, inner_train_labels, load_fn=volume_cache.get)
    inner_val_ds = dataset.DatParkinsonDataset(inner_val_uids, inner_val_labels, load_fn=volume_cache.get)
    inner_train_loader = torch.utils.data.DataLoader(inner_train_ds, batch_size=batch_size, shuffle=True, num_workers=0)
    inner_val_loader = torch.utils.data.DataLoader(inner_val_ds, batch_size=batch_size, num_workers=0)

    torch.manual_seed(seed)
    net = model.build_model().to(config.DEVICE)
    optimizer = torch.optim.Adam(net.parameters(), lr=lr, weight_decay=config.WEIGHT_DECAY)
    loss_fn = torch.nn.BCEWithLogitsLoss()

    best_state, history = train_mod.train_one_fold(
        net, inner_train_loader, inner_val_loader, optimizer, loss_fn,
        epochs=epochs, patience=patience, device=config.DEVICE,
        use_amp=config.USE_AMP, seed=seed,
    )
    net.load_state_dict(best_state)

    outer_ds = dataset.DatParkinsonDataset(outer_uids, load_fn=volume_cache.get)
    outer_loader = torch.utils.data.DataLoader(outer_ds, batch_size=batch_size, num_workers=0)
    outer_probs = []
    for x, _ in outer_loader:
        outer_probs.append(model.predict(net, x))
    return np.concatenate(outer_probs), history, best_state

In [5]:
# [RUN ME] -- fold-0 sanity check (no hyperparameter search -- Boulkrinat
# et al.'s literal patch_size/patch_distance values, already the
# defaults in data.denoise_volume).
batch_size, lr = 32, 2e-3  # rung 2/3's validated winner, held fixed here

outer_folds = evaluate.make_folds(np.array(labels), np.array(families),
                                   n_splits=config.N_FOLDS, random_state=config.SEED)
fold0_train_idx, fold0_test_idx = outer_folds[0]
fold0_train_uids = [uids[i] for i in fold0_train_idx]
fold0_train_labels = [labels[i] for i in fold0_train_idx]
fold0_train_family = [families[i] for i in fold0_train_idx]
fold0_test_uids = [uids[i] for i in fold0_test_idx]
fold0_test_labels = np.array([labels[i] for i in fold0_test_idx])

start = time.time()
probs, history, best_state = train_and_score_nested(
    fold0_train_uids, fold0_train_labels, fold0_train_family,
    fold0_test_uids, batch_size=batch_size, lr=lr, seed=config.SEED,
)
elapsed = time.time() - start
score = evaluate.log_loss_score(fold0_test_labels, probs)
print(f"fold 0 sanity check: {len(history['val_loss'])} epochs, inner-val best="
      f"{min(history['val_loss']):.4f}, outer log loss={score:.4f}, {elapsed:.1f}s "
      f"({elapsed / len(history['val_loss']):.2f}s/epoch)")
print("rung 3's fold-0/seed-42 log loss for comparison: 0.4005 (README.md)")

fold 0 sanity check: 26 epochs, inner-val best=0.3722, outer log loss=0.3614, 24.5s (0.94s/epoch)
rung 3's fold-0/seed-42 log loss for comparison: 0.4005 (README.md)


In [6]:
# [RUN ME] -- full 5-fold nested CV, repeated 5x, on the denoised cache.
# Same protocol as rung 3 (notebooks/07_cnn_rung3.ipynb).
N_REPEATS = 5
oof_repeats_denoise = []

for repeat_seed in range(config.SEED, config.SEED + N_REPEATS):
    outer_folds = evaluate.make_folds(np.array(labels), np.array(families),
                                       n_splits=config.N_FOLDS, random_state=repeat_seed)
    oof_probs = np.zeros(len(uids))
    for fold_i, (train_idx, test_idx) in enumerate(outer_folds):
        fold_train_uids = [uids[i] for i in train_idx]
        fold_train_labels = [labels[i] for i in train_idx]
        fold_train_family = [families[i] for i in train_idx]
        fold_test_uids = [uids[i] for i in test_idx]

        probs, history, best_state = train_and_score_nested(
            fold_train_uids, fold_train_labels, fold_train_family,
            fold_test_uids, batch_size=batch_size, lr=lr, seed=repeat_seed,
        )
        oof_probs[test_idx] = probs
        torch.save(best_state, config.CHECKPOINT_DIR / f"rung4_denoise_seed{repeat_seed}_fold{fold_i}.pt")
        fold_score = evaluate.log_loss_score(np.array(labels)[test_idx], probs)
        print(f"  seed={repeat_seed} fold={fold_i}: {len(history['val_loss'])} epochs, "
              f"outer fold log loss={fold_score:.4f}")

    repeat_logloss = evaluate.log_loss_score(np.array(labels), oof_probs)
    oof_repeats_denoise.append(oof_probs)
    print(f"seed={repeat_seed} pooled OOF log loss: {repeat_logloss:.4f}")
    np.save(config.DATA_PROCESSED / f"rung4_denoise_oof_seed{repeat_seed}.npy", oof_probs)

repeat_scores_denoise = np.array([evaluate.log_loss_score(np.array(labels), oof) for oof in oof_repeats_denoise])
print(f"\n{N_REPEATS}-repeat denoised CNN pooled log loss: "
      f"mean={repeat_scores_denoise.mean():.4f}, sd={repeat_scores_denoise.std(ddof=1):.4f}")
print("current validated CNN (rung 3, README.md 2026-09-09): mean=0.4520, sd=0.0109")

  seed=42 fold=0: 26 epochs, outer fold log loss=0.3614
  seed=42 fold=1: 25 epochs, outer fold log loss=0.4535
  seed=42 fold=2: 21 epochs, outer fold log loss=0.4276
  seed=42 fold=3: 17 epochs, outer fold log loss=0.4419
  seed=42 fold=4: 32 epochs, outer fold log loss=0.4639
seed=42 pooled OOF log loss: 0.4296
  seed=43 fold=0: 23 epochs, outer fold log loss=0.4487
  seed=43 fold=1: 15 epochs, outer fold log loss=0.4782
  seed=43 fold=2: 27 epochs, outer fold log loss=0.4094
  seed=43 fold=3: 28 epochs, outer fold log loss=0.4504
  seed=43 fold=4: 29 epochs, outer fold log loss=0.4863
seed=43 pooled OOF log loss: 0.4546
  seed=44 fold=0: 34 epochs, outer fold log loss=0.4037
  seed=44 fold=1: 19 epochs, outer fold log loss=0.4973
  seed=44 fold=2: 21 epochs, outer fold log loss=0.3728
  seed=44 fold=3: 19 epochs, outer fold log loss=0.4295
  seed=44 fold=4: 30 epochs, outer fold log loss=0.4748
seed=44 pooled OOF log loss: 0.4356
  seed=45 fold=0: 21 epochs, outer fold log loss=0.4

In [8]:
# [RUN ME] (no data access itself). Gate: paired per-repeat delta vs.
# the current validated CNN, using evaluate.paired_repeat_gate (fixed
# 2026-09-10 per Opus review -- the old gate here used
# paired_bootstrap_ci on a single arbitrarily-chosen repeat plus a noise
# threshold scaled by a single repeat's sd instead of the mean's
# standard error; see project memory
# project_dat_parkinson_rung4_gate_review.md).
y_true = np.array(labels)
repeat_seeds = list(range(config.SEED, config.SEED + N_REPEATS))
current_cnn_oof_by_repeat = [
    np.load(config.DATA_PROCESSED / f"rung3_oof_seed{s}.npy") for s in repeat_seeds
]

deltas = [
    evaluate.log_loss_score(y_true, oof_repeats_denoise[i]) - evaluate.log_loss_score(y_true, current_cnn_oof_by_repeat[i])
    for i in range(N_REPEATS)
]
gate = evaluate.paired_repeat_gate(deltas)

print(f"per-repeat deltas (denoised - current CNN): {[f'{d:+.4f}' for d in deltas]}")
print(f"mean={gate['mean']:+.4f}, sd={gate['sd']:.4f}, "
      f"95% CI=[{gate['ci_low']:+.4f}, {gate['ci_high']:+.4f}]")
print(f"GATE {'PASSED' if gate['passed'] else 'NOT PASSED'}: "
      f"{'denoising REPLACES the current CNN.' if gate['passed'] else 'does not beat the current CNN by more than noise -- keep the current CNN.'}")
if gate["passed"]:
    print("\nREMINDER: if adopted, config.USE_NLM_DENOISING=True must also be set "
          "for submission_src/main.py's inference path, and the ~"
          f"{mean_denoise * 1000:.0f}ms/volume denoising cost (cell 2) must be "
          "budgeted against the submission's 3-hour limit alongside the CNN "
          "ensemble's own cost.")

per-repeat deltas (denoised - current CNN): ['-0.0279', '-0.0052', '+0.0008', '-0.0098', '+0.0291']
mean=-0.0026, sd=0.0207, 95% CI=[-0.0284, +0.0231]
GATE NOT PASSED: does not beat the current CNN by more than noise -- keep the current CNN.


**What we're looking for:** does NL-means denoising (in place of no
denoising in rungs 0-3) beat the current validated CNN (0.4520) by more
than noise? This is the last of the 5 rung-4 experiments.

**What we found:** denoising adds 4.1x the resample+crop cost per
volume (1100ms vs. 271ms, n=15 sample); the estimated ~31min full cache
rebuild undershot the real one, which took 3984.5s (~66 min) — worth
remembering if this or a similar preprocessing change is ever timed
again. Fold-0 sanity check was promising in isolation (26 epochs, outer
log loss 0.3614 vs. rung 3's 0.4005 on the same fold/seed), but didn't
hold up over the full 5x5: 5-repeat pooled mean=0.4494, sd=0.0184.
Per-repeat deltas (denoised − current CNN) = [-0.0279, -0.0052, +0.0008,
-0.0098, +0.0291], mean=-0.0026, sd=0.0207 (the widest of any rung-4
experiment), 95% CI=[-0.0284, +0.0231] comfortably straddling zero.
**GATE NOT PASSED.**

**Decision / next step:** the fold-0 result did not generalize — the
5-repeat mean is essentially flat and the per-repeat delta's sign flips
twice, so this reads as a genuine negative rather than an underpowered
one (contrast experiments 2/3, both consistently directionally
positive). Keep the current CNN and current `data.py` behavior
(`USE_NLM_DENOISING` stays `False` everywhere, including
`submission_src/main.py`); `rung4_denoise_*` checkpoints not used.
**This closes out rung 4: 0/5 experiments passed the gate.** The
rung-3 CNN + classical-baseline blend (`w_cnn=0.70`, real submission
log loss 0.4648) remains the production model. Logged in `README.md`'s
Progress section (2026-09-10). If time allows before 2026-09-16, the
Opus review's highest-value follow-up is re-running experiments 2+3
combined at a higher repeat count — see project memory
`project_dat_parkinson_rung4_gate_review.md`.